# Tesis MIA 303 — Exploración inicial con DuckDB**Autor:** Carlos Pérez Pérez · **Curso:** MIA 303 – Proyectos de Investigación I  **Docente:** Mg. Eng. Maria F. Tejada Begazo · **Mayo 2026**---Este notebook conecta DuckDB a los datasets **MIMIC-IV v3.1** (módulos `hosp` e `icu`) y **MIMIC-IV-Note v2.2** (notas clínicas no estructuradas) para realizar la exploración inicial del corpus que sustentará la propuesta de tesis.## Estructura del notebook1. **Pre-requisitos y configuración** — instala dependencias y verifica rutas2. **Conexión DuckDB** — crea la base de trabajo y registra los archivos como vistas3. **Los 5 patrones esenciales** — los únicos que necesitarás para toda la tesis4. **Exploración inicial** — primer vistazo al volumen y características del corpus5. **Próximos pasos** — siguiente sprint hacia la Etapa 1 NLP---> **Marco ético:** los datos utilizados son MIMIC-IV v3.1 (DOI 10.13026/kpb9-mt58) y MIMIC-IV-Note v2.2 (DOI 10.13026/1n74-ne17), ambos accedidos bajo PhysioNet Credentialed Health Data Use Agreement v1.5.0 firmado por el investigador. No se redistribuyen datos, no se intenta re-identificar pacientes y todos los análisis se realizan localmente.

## 1. Pre-requisitos y configuración### 1.1 LibreríasSi todavía no instalaste las dependencias, ejecuta la celda siguiente. Si ya lo hiciste en Git Bash, puedes saltarla.

In [ ]:
# Instalación (solo la primera vez)# !pip install duckdb pandas pyarrow tqdm matplotlib --quiet

In [ ]:
import duckdbimport pandas as pdfrom pathlib import Pathimport os, sys, gzipprint("Python      :", sys.version.split()[0])print("DuckDB      :", duckdb.__version__)print("Pandas      :", pd.__version__)

### 1.2 Configuración de rutasAjusta las dos rutas siguientes si tus archivos están en otro lugar. Por defecto asumen la estructura que acordamos:```C:\MIMIC\├── mimiciv\          # MIMIC-IV principal (cuando lo descargues)│   ├── hosp\│   └── icu\└── note\             # MIMIC-IV-Note (la que estás descargando ahora)    └── physionet.org\files\mimic-iv-note\2.2\note\```

In [ ]:
# Ajusta solo estas dos rutas si las tuyas son diferentesPATH_MIMIC_MAIN = Path(r"C:\MIMIC\mimiciv")PATH_MIMIC_NOTE = Path(r"C:\MIMIC\note\physionet.org\files\mimic-iv-note\2.2\note")# Carpeta de trabajo para la base de DuckDB y salidasPATH_WORK = Path(r"C:\MIMIC\tesis")PATH_WORK.mkdir(parents=True, exist_ok=True)DB_FILE = PATH_WORK / "mimic.db"print(f"Main:  {PATH_MIMIC_MAIN}  (existe: {PATH_MIMIC_MAIN.exists()})")print(f"Note:  {PATH_MIMIC_NOTE}  (existe: {PATH_MIMIC_NOTE.exists()})")print(f"Work:  {PATH_WORK}")print(f"DB:    {DB_FILE}")

### 1.3 Verificación de archivosEsta celda comprueba qué tienes disponible. No falla si algo falta; solo te avisa para que sepas qué bloques del notebook puedes ejecutar.

In [ ]:
def check(path, label):    """Comprueba si un archivo existe y muestra su tamaño."""    if path.exists():        size_mb = path.stat().st_size / 1024 / 1024        print(f"  [OK]    {label:35s}  ({size_mb:7.1f} MB)")        return True    else:        print(f"  [FALTA] {label:35s}")        return Falseprint("MIMIC-IV principal (módulo hosp):")check(PATH_MIMIC_MAIN / "hosp" / "admissions.csv.gz", "admissions.csv.gz")check(PATH_MIMIC_MAIN / "hosp" / "patients.csv.gz", "patients.csv.gz")check(PATH_MIMIC_MAIN / "hosp" / "diagnoses_icd.csv.gz", "diagnoses_icd.csv.gz")check(PATH_MIMIC_MAIN / "hosp" / "labevents.csv.gz", "labevents.csv.gz")print("\nMIMIC-IV principal (módulo icu):")check(PATH_MIMIC_MAIN / "icu" / "icustays.csv.gz", "icustays.csv.gz")check(PATH_MIMIC_MAIN / "icu" / "chartevents.csv.gz", "chartevents.csv.gz")print("\nMIMIC-IV-Note (cuando termine la descarga):")check(PATH_MIMIC_NOTE / "discharge.csv.gz", "discharge.csv.gz")check(PATH_MIMIC_NOTE / "discharge_detail.csv.gz", "discharge_detail.csv.gz")check(PATH_MIMIC_NOTE / "radiology.csv.gz", "radiology.csv.gz")check(PATH_MIMIC_NOTE / "radiology_detail.csv.gz", "radiology_detail.csv.gz")

## 2. Conexión DuckDBDuckDB es un motor de base de datos analítico ultraligero. No necesita servicio corriendo, no necesita instalar un servidor: es solo una librería de Python.La siguiente celda crea (o reabre si ya existe) un archivo `mimic.db` en `C:\MIMIC\tesis\` y registra cada archivo `.csv.gz` como una **vista**. Una vista en DuckDB es un puntero al archivo — **no copia los datos** y por eso es instantánea de crear.

In [ ]:
con = duckdb.connect(str(DB_FILE))print(f"Conectado a {DB_FILE}")print(f"Versión DuckDB: {duckdb.__version__}\n")# --- Registrar vistas de MIMIC-IV-Note ---if (PATH_MIMIC_NOTE / "discharge.csv.gz").exists():    con.execute(f"""        CREATE OR REPLACE VIEW discharge AS        SELECT * FROM read_csv_auto('{(PATH_MIMIC_NOTE / "discharge.csv.gz").as_posix()}', compression='gzip')    """)    print("Vista 'discharge' creada.")else:    print("[Pendiente] discharge.csv.gz aún no disponible.")if (PATH_MIMIC_NOTE / "discharge_detail.csv.gz").exists():    con.execute(f"""        CREATE OR REPLACE VIEW discharge_detail AS        SELECT * FROM read_csv_auto('{(PATH_MIMIC_NOTE / "discharge_detail.csv.gz").as_posix()}', compression='gzip')    """)    print("Vista 'discharge_detail' creada.")# --- Registrar vistas de MIMIC-IV principal (hosp) si están disponibles ---hosp_tables = ["admissions", "patients", "diagnoses_icd", "d_icd_diagnoses",               "labevents", "prescriptions"]for tbl in hosp_tables:    p = PATH_MIMIC_MAIN / "hosp" / f"{tbl}.csv.gz"    if p.exists():        con.execute(f"""            CREATE OR REPLACE VIEW {tbl} AS            SELECT * FROM read_csv_auto('{p.as_posix()}', compression='gzip')        """)        print(f"Vista '{tbl}' creada.")# --- Registrar vistas de MIMIC-IV principal (icu) ---icu_tables = ["icustays", "chartevents", "d_items"]for tbl in icu_tables:    p = PATH_MIMIC_MAIN / "icu" / f"{tbl}.csv.gz"    if p.exists():        con.execute(f"""            CREATE OR REPLACE VIEW {tbl} AS            SELECT * FROM read_csv_auto('{p.as_posix()}', compression='gzip')        """)        print(f"Vista '{tbl}' creada.")# Listar vistas activasprint("\nVistas registradas en la base:")display(con.execute("SHOW TABLES").df())

## 3. Los 5 patrones que necesitarás para toda la tesisA partir de aquí, **todo lo que haces es combinar estos 5 patrones**. No tienes que aprender más SQL del que se ve abajo.> **Truco importante:** cada query devuelve un objeto DuckDB. Para convertirlo a un DataFrame de pandas (que ya sabes manejar), le agregas `.df()` al final. Listo.

### Patrón 1 — Contar filasÚtil para entender el volumen del corpus sin cargar nada en memoria.

In [ ]:
# ¿Cuántas epicrisis (discharge summaries) hay en MIMIC-IV-Note?con.execute("SELECT COUNT(*) AS total_notas FROM discharge").df()

### Patrón 2 — Filtrar por una condiciónÚtil para quedarte solo con un subconjunto de interés.

In [ ]:
# Notas de discharge correspondientes al servicio de Medicina (ejemplo)# La columna 'note_type' indica el tipo de notacon.execute("""    SELECT note_type, COUNT(*) AS n    FROM discharge    GROUP BY note_type    ORDER BY n DESC""").df()

### Patrón 3 — Cruzar tablas (JOIN)Útil para enriquecer las notas con metadata de las hospitalizaciones (ej. length of stay, diagnósticos).

In [ ]:
# Si tienes MIMIC-IV principal descargado, esta query une discharge con admissions# para obtener la duración de la estancia (length of stay)# Si todavía no tienes 'admissions', esta celda mostrará un error: ignóralo por ahora.try:    con.execute("""        SELECT d.subject_id, d.hadm_id, d.note_type,               a.admittime, a.dischtime,               DATE_DIFF('hour', a.admittime, a.dischtime) AS los_hours        FROM discharge d        JOIN admissions a USING (hadm_id)        LIMIT 5    """).df()except Exception as e:    print(f"[Skip] MIMIC-IV principal todavía no disponible — esta query corre después.")    print(f"       Mensaje técnico: {str(e)[:120]}")

### Patrón 4 — Tomar una muestra aleatoriaImprescindible para inspección manual y construcción del corpus etiquetado de la Etapa 1.

In [ ]:
# 10 notas tomadas al azar para inspecciónsample = con.execute("""    SELECT subject_id, hadm_id, note_type, charttime, text    FROM discharge    USING SAMPLE 10""").df()print(f"Muestra de {len(sample)} notas:")sample[['subject_id', 'hadm_id', 'note_type', 'charttime']]

### Patrón 5 — Convertir a DataFrame para seguir trabajando como siempreCuando termines de filtrar con DuckDB, llamas `.df()` y de ahí en adelante trabajas con pandas, scikit-learn, transformers, etc.

In [ ]:
# Ejemplo: tomar 100 notas y trabajar con la primera en pandasnotas_100 = con.execute("SELECT * FROM discharge USING SAMPLE 100").df()print(f"Tipo: {type(notas_100)}")print(f"Forma: {notas_100.shape}")print(f"Columnas: {list(notas_100.columns)}")print("\nPrimeros 800 caracteres de la primera nota:")print("-" * 60)print(notas_100.iloc[0]['text'][:800])

## 4. Exploración inicial del corpusCuatro queries de descubrimiento. Cada una responde una pregunta que vas a citar en el Capítulo III de la tesis.

### 4.1 ¿Cuántas notas y de qué tipos?

In [ ]:
resumen_tipos = con.execute("""    SELECT note_type,           COUNT(*) AS n_notas,           COUNT(DISTINCT subject_id) AS n_pacientes,           COUNT(DISTINCT hadm_id) AS n_hospitalizaciones    FROM discharge    GROUP BY note_type    ORDER BY n_notas DESC""").df()print("Distribución de notas por tipo:")resumen_tipos

### 4.2 ¿Qué tan largas son las notas?La longitud impacta directamente el costo computacional del NLP. BioBERT y ClinicalBERT tienen un máximo de 512 tokens por entrada, así que necesitamos saber cuántas notas exceden ese límite y planear segmentación.

In [ ]:
longitud = con.execute("""    SELECT        MIN(LENGTH(text)) AS min_chars,        AVG(LENGTH(text)) AS avg_chars,        MEDIAN(LENGTH(text)) AS median_chars,        MAX(LENGTH(text)) AS max_chars,        AVG(LENGTH(text) - LENGTH(REPLACE(text, ' ', ''))) AS avg_palabras_aprox    FROM discharge""").df()print("Estadísticas de longitud de las notas:")longitud.T  # Transponer para que sea más fácil de leer

### 4.3 Estimación rápida de tokensUna regla heurística: 1 token ≈ 4 caracteres en inglés clínico. Cualquier nota con más de 2048 caracteres muy probablemente exceda los 512 tokens del modelo.

In [ ]:
distribucion_tokens = con.execute("""    SELECT        CASE            WHEN LENGTH(text) < 2048  THEN '< 512 tokens (cabe sin segmentar)'            WHEN LENGTH(text) < 8192  THEN '512-2048 tokens (segmentar en 2-4 ventanas)'            WHEN LENGTH(text) < 32768 THEN '2048-8192 tokens (segmentar en 4-16)'            ELSE                            '> 8192 tokens (caso extremo)'        END AS rango_tokens,        COUNT(*) AS n_notas    FROM discharge    GROUP BY rango_tokens    ORDER BY n_notas DESC""").df()print("Distribución estimada de notas por longitud en tokens:")distribucion_tokens

### 4.4 Sample de 100 notas para empezar el desarrollo NLPEste es el corpus inicial que vas a usar para probar BioBERT/ClinicalBERT en la Fase 2 del roadmap. Lo guardamos como CSV separado dentro de tu carpeta de trabajo (no contiene PHI gracias a la deidentificación de MIMIC, pero sigue bajo DUA — no lo subas a GitHub).

In [ ]:
# Sample reproducible (seed fijo para que cada vez salga el mismo conjunto)sample_dev = con.execute("""    SELECT subject_id, hadm_id, note_type, charttime, text    FROM discharge    USING SAMPLE 100 ROWS (reservoir, 42)""").df()# Guardar para experimentos siguientesout_path = PATH_WORK / "sample_100_discharge.csv"sample_dev.to_csv(out_path, index=False)print(f"Guardado: {out_path}")print(f"Tamaño: {out_path.stat().st_size / 1024:.1f} KB")print(f"Notas en el sample: {len(sample_dev)}")print(f"Pacientes únicos: {sample_dev['subject_id'].nunique()}")print(f"Hospitalizaciones únicas: {sample_dev['hadm_id'].nunique()}")

## 5. Próximos pasos del roadmap experimentalCon este notebook validaste que:- [x] DuckDB está conectado a tus datos- [x] Puedes contar, filtrar, cruzar, samplear y exportar- [x] Tienes una muestra inicial de 100 notas en `sample_100_discharge.csv`El siguiente notebook (Fase 2 del roadmap) será **`fase2_nlp_bioBERT.ipynb`**, donde:1. Cargas BioBERT y ClinicalBERT con `transformers`2. Tokenizas el sample de 100 notas3. Ejecutas NER de entidades médicas (eventos adversos)4. Comparas resultados con la taxonomía del Anexo 02 de la Directiva GG-ESSALUD-20215. Calculas las métricas iniciales (precision, recall, F1)Antes de ese paso, sugiero:1. **Validar la deidentificación**: confirmar que las notas no contienen PHI residual. Spot check de 5-10 notas del sample.2. **Construir el etiquetado inicial**: anotar manualmente eventos adversos en 20 notas del sample, para tener verdad-fundamento de comparación.3. **Si todavía no descargaste MIMIC-IV principal**, hazlo en paralelo: lo necesitarás para la Etapa 2 (priorización GEMSES) que usa las variables proxy tabulares.---> **Recordatorio ético:** este notebook y todos sus outputs (incluyendo `sample_100_discharge.csv`) están sujetos al DUA de PhysioNet. No subir a GitHub público, no compartir con terceros que no hayan firmado el DUA, no intentar re-identificar pacientes.

In [ ]:
# Cerrar la conexión cuando termines la sesióncon.close()print("Conexión cerrada. La base 'mimic.db' queda guardada en disco.")